# Calogero-Sutherland Model — Exact Wavefunction Test

**Hamiltonian** (convention $\hbar^2/m = 1$, so kinetic $= -\sum_i \partial^2/\partial x_i^2$):

$$H = -\sum_i \frac{\partial^2}{\partial x_i^2} + \sum_i x_i^2 + 2L(L-1) \sum_{i<j} \frac{1}{(x_i - x_j)^2}$$

**Exact ground state:**

$$\psi_0(x) = C \prod_{i<j} |x_i - x_j|^L \cdot \exp\!\left(-\tfrac{1}{2}\sum_i x_i^2\right)$$

$$\log|\psi_0| = L \sum_{i<j} \log|x_i - x_j| - \tfrac{1}{2}\sum_i x_i^2$$

**Exact ground state energy** (in code convention, $\omega=1$):

$$E_0 = N\bigl(1 + L(N-1)\bigr) \qquad \Longrightarrow \qquad E_0/N = 1 + L(N-1)$$

**This notebook trains the exact model** — a single learnable parameter $\lambda$ in the Jastrow factor.
When $\lambda \to L$, the energy converges exactly to $E_0$.

## Imports

In [1]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt

from qvarnet.train import train
from qvarnet.config.training_setup import TrainingConfig
from qvarnet.config.coord_mode import LabCoords
from qvarnet.models.analytic import CalogeroSutherlandAnalyticModel
from qvarnet.models.exponential import JastrowLogExponentialMLPwithGaussianPenalty, LogExponentialMLPwithGaussianPenalty
from qvarnet.hamiltonian.continuous import CalogeroSutherlandHamiltonian

## System

In [2]:
N_PARTICLES = 4      # number of particles
DIM         = 1
L           = 1.8   # CS coupling; exact λ = L in the Jastrow factor

N_CHAINS = 5_000
DoF      = N_PARTICLES * DIM
SHAPE    = (N_CHAINS, DoF)

# Exact energy in code convention (H = -Σ∂² + V, ω=1)
E_EXACT_TOTAL = N_PARTICLES * (1 + L * (N_PARTICLES - 1))
E_EXACT       = E_EXACT_TOTAL / N_PARTICLES   # per particle

print(f"N = {N_PARTICLES},  L = {L}")
print(f"E₀ (total)      = {E_EXACT_TOTAL:.4f}")
print(f"E₀ (per particle) = {E_EXACT:.4f}")

N = 4,  L = 1.8
E₀ (total)      = 25.6000
E₀ (per particle) = 6.4000


## Model

The `CalogeroSutherlandAnalyticModel` implements the exact log-wavefunction with one learnable parameter $\lambda$:

$$\log|\psi| = \lambda \sum_{i<j} \log|x_i - x_j| - \tfrac{1}{2}\sum_i x_i^2$$

Setting `lambda_init = L` starts at the exact solution. Any other starting value tests convergence.

In [3]:
LAMBDA_INIT = 1.7   # starting guess for λ; converges to L during training

# model        = CalogeroSutherlandAnalyticModel(lambda_init=LAMBDA_INIT)
model = LogExponentialMLPwithGaussianPenalty(architecture = [N_PARTICLES, 100, 1])
# model = JastrowLogExponentialMLPwithGaussianPenalty(architecture = [N_PARTICLES, 1], lambda_init=LAMBDA_INIT)
IS_LOG_MODEL = True

print(f"λ init = {LAMBDA_INIT}   →   target λ = {L}")

λ init = 1.7   →   target λ = 1.8


## Hamiltonian

`epsilon` softens the $1/(x_i-x_j)^2$ singularity; set it small but nonzero for numerical stability.

In [4]:
hamiltonian = CalogeroSutherlandHamiltonian(L=L, epsilon=1e-8)

## Optimizer

In [5]:
LEARNING_RATE = 0.01
optimizer     = optax.adam(learning_rate=LEARNING_RATE)

## Sampler

In [6]:
CHAIN_LENGTH = 10

sampler_params = {
    "step_size":            0.5,
    "chain_length":         CHAIN_LENGTH + 1,
    "thermalization_steps": CHAIN_LENGTH,
    "thinning_factor":      1,
    "PBC":                  1.0,
}

## Training

In [10]:
def train_helper(model, epochs, seed1, seed2, with_cusp_condition, n_configs_per_pair):
    cfg = TrainingConfig(
        n_epochs=epochs,
        rng_seed=seed1,
        warm_walkers=True,
        is_update_step_size=True,
        # is_log_model=IS_LOG_MODEL,
        min_step=1e-5,
        max_step=5.0,
        use_cusp_condition=with_cusp_condition,
        cusp_epsilon=1e-4,
        cusp_n_configs_per_pair=n_configs_per_pair,
        cusp_rng_seed=seed2,
        cusp_alpha=.1,
        cusp_n= 2, # exponent of the cusp potential term. n -> 1/r^n
        cusp_C_n=L, # expected value of the cusp condition coefficient C_n; used to set the strength of the cusp potential term
    )

    history,_,_ = train(
        shape=SHAPE,
        model=model,
        optimizer=optimizer,
        hamiltonian=hamiltonian,
        training_config=cfg,
        sampler_params=sampler_params,
        coord_mode=LabCoords(),
    )
    return history

In [11]:
def analyze_and_plot(history):
    energies    = np.array([float(s.energy)   for s in history]) / N_PARTICLES
    stds       = np.array([float(s.std)    for s in history]) / N_PARTICLES
    acc_rates  = np.array([float(jnp.mean(s.acceptance_rate)) for s in history])
    step_sizes = np.array([float(s.step_size) for s in history])
    if history[-1].params["params"].get("lam") is not None:
        lambdas    = np.array([float(s.params["params"]["lam"]) for s in history])
    steps      = np.arange(len(history))

    TAIL = 200
    final_E   = energies[-TAIL:].mean()
    final_std = stds[-TAIL:].mean()
    if history[-1].params["params"].get("lam") is not None:
        final_lam = lambdas[-1]

    print(f"L (exact coupling)     : {L}")
    if history[-1].params["params"].get("lam") is not None:
        print(f"λ final                : {final_lam:.6f}   (target: {L})")
        print(f"λ error                : {abs(final_lam - L):.2e}")
    print()
    print(f"E₀/N (exact)           : {E_EXACT:.6f}")
    print(f"E/N  (last {TAIL})       : {final_E:.6f} ± {final_std:.6f}")

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(
        f"Calogero-Sutherland  N={N_PARTICLES}  L={L}  "
        f"(exact model: single parameter λ → L)",
        fontsize=13,
    )
    print(len(energies))

    ax = axes[0, 0]
    ax.plot(steps, energies, lw=0.8, label='E/N')
    ax.fill_between(steps, energies - stds, energies + stds, alpha=0.3, label='±σ')
    ax.axhline(E_EXACT, color='red', ls='--', lw=1.2, label=f'Exact E₀/N = {E_EXACT:.4f}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('E/N'); ax.set_title('Energy'); ax.legend(); ax.set_ylim(E_EXACT, 10)

    ax = axes[0, 1]
    ax.semilogy(steps, np.abs(energies - E_EXACT), lw=0.8, color='C1')
    ax.set_xlabel('Epoch'); ax.set_ylabel('|E/N − E₀/N|'); ax.set_title('Energy error (log)')

    ax = axes[0, 2]
    ax.semilogy(steps, stds, lw=0.8, color='C2')
    ax.set_xlabel('Epoch'); ax.set_ylabel('σ(E)/N'); ax.set_title('Energy std (log)')

    if history[-1].params["params"].get("lam") is not None:
        ax = axes[1, 0]
        ax.plot(steps, lambdas, lw=0.8, color='C3', label='λ(t)')
        ax.axhline(L, color='red', ls='--', lw=1.2, label=f'L = {L}')
        ax.set_xlabel('Epoch'); ax.set_ylabel('λ'); ax.set_title('Jastrow exponent λ → L'); ax.legend()

    ax = axes[1, 1]
    ax.plot(steps, acc_rates, lw=0.8, color='C4')
    ax.axhline(0.5, color='red', ls='--', label='target 0.5')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Acceptance rate'); ax.set_title('MH acceptance rate'); ax.legend()

    ax = axes[1, 2]
    ax.plot(steps, step_sizes, lw=0.8, color='C5')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Step size'); ax.set_title('MH step size')

    plt.tight_layout()
    plt.savefig('cs_model_training.png', dpi=150, bbox_inches='tight')
    plt.show()

# Study of the convergence of the energy with and without the cusp condition in the loss

## Cusp condition enabled

In [12]:
analytic_model        = CalogeroSutherlandAnalyticModel(lambda_init=LAMBDA_INIT)
mlp_gaussian          = LogExponentialMLPwithGaussianPenalty(architecture = [N_PARTICLES, 100, 1])
mlp_jastrow_gaussian  = JastrowLogExponentialMLPwithGaussianPenalty(architecture = [N_PARTICLES, 1], lambda_init=LAMBDA_INIT)

models = {
    "Analytic": analytic_model,
    "MLP+Gaussian": mlp_gaussian,
    "MLP+Jastrow+Gaussian": mlp_jastrow_gaussian,
}

n_configs_per_pair = [0, 1, 2]


class ResultsCollector:
    def __init__(self):
        # model_name -> n_configs -> list of run dicts
        self.results = {}

    def add_result(self, model_name, n_configs, history, with_cusp):
        if model_name not in self.results:
            self.results[model_name] = {}
        if n_configs not in self.results[model_name]:
            self.results[model_name][n_configs] = []
        self.results[model_name][n_configs].append({
            "energy":          [float(s.energy) for s in history],
            "std":             [float(s.std) for s in history],
            "acceptance_rate": [float(jnp.mean(s.acceptance_rate)) for s in history],
            "step_size":       [float(s.step_size) for s in history],
            "cusp":            with_cusp,
            "fin_params":      history[-1].params,
        })


collector = ResultsCollector()


for model_name, model in models.items():
    for n_configs in n_configs_per_pair:
        print(n_configs)
        current_seeds = np.random.randint(0, 10000, size=(2, 2))
        if n_configs == 0:
            with_cusp = False  # only cusp condition runs when n_configs=0
        else:
            with_cusp = True
        for seed_tuple in current_seeds:
            print(f"Training {model_name} | n_configs={n_configs} | cusp={with_cusp} | seeds={seed_tuple}")
            history = train_helper(
                model=model, epochs=500,
                seed1=seed_tuple[0], seed2=seed_tuple[1],
                with_cusp_condition=with_cusp,
                n_configs_per_pair=n_configs,
            )
            collector.add_result(model_name, n_configs, history, with_cusp)


0
Training Analytic | n_configs=0 | cusp=False | seeds=[5172 1368]


TypeError: TrainingConfig.__init__() got an unexpected keyword argument 'use_cusp_condition'

In [15]:
for model_name, model_results in collector.results.items():
    print(f"Model: {model_name}")
    if model_name == "Analytic":
        for n_configs, results in model_results.items():
            print(n_configs, "n_configs")
            print(results[0])
            print()

Model: Analytic
0 n_configs
{'energy': [27.670543670654297, 25.898767471313477, 25.686702728271484, 25.629192352294922, 25.65157699584961, 25.611270904541016, 25.624353408813477, 25.626136779785156, 25.61545753479004, 25.60480499267578, 25.612552642822266, 25.609560012817383, 25.601303100585938, 25.60462188720703, 25.6054744720459, 25.60431480407715, 25.600698471069336, 25.599313735961914, 25.598209381103516, 25.604663848876953, 25.59744644165039, 25.600252151489258, 25.60173988342285, 25.60021209716797, 25.600595474243164, 25.600051879882812, 25.599966049194336, 25.600128173828125, 25.600379943847656, 25.600515365600586, 25.60068130493164, 25.600589752197266, 25.600364685058594, 25.600187301635742, 25.60048484802246, 25.60102081298828, 25.598529815673828, 25.600961685180664, 25.599687576293945, 25.600515365600586, 25.601961135864258, 25.59583282470703, 25.600749969482422, 25.600387573242188, 25.600069046020508, 25.600034713745117, 25.599821090698242, 25.599124908447266, 25.60104370117

In [10]:
history = train_helper(epochs=500, seed1=42, seed2=43, with_cusp_condition=False, n_configs_per_pair=2)
analyze_and_plot(history)

TypeError: train_helper() missing 1 required positional argument: 'model'

## Extract History

In [ ]:
energies   = np.array([float(s.energy) for s in history]) / N_PARTICLES
stds       = np.array([float(s.std)    for s in history]) / N_PARTICLES
acc_rates  = np.array([float(jnp.mean(s.acceptance_rate)) for s in history])
step_sizes = np.array([float(s.step_size) for s in history])
if history[-1].params["params"].get("lam") is not None:
    lambdas    = np.array([float(s.params["params"]["lam"]) for s in history])
steps      = np.arange(len(history))

## Summary

In [ ]:
TAIL = 200
final_E   = energies[-TAIL:].mean()
final_std = stds[-TAIL:].mean()
if history[-1].params["params"].get("lam") is not None:
    final_lam = lambdas[-1]

print(f"L (exact coupling)     : {L}")
if history[-1].params["params"].get("lam") is not None:
    print(f"λ final                : {final_lam:.6f}   (target: {L})")
    print(f"λ error                : {abs(final_lam - L):.2e}")
print()
print(f"E₀/N (exact)           : {E_EXACT:.6f}")
print(f"E/N  (last {TAIL})       : {final_E:.6f} ± {final_std:.6f}")
print(f"|E - E₀|/N             : {abs(final_E - E_EXACT):.2e}")
print(f"Relative error         : {abs(final_E - E_EXACT) / E_EXACT * 100:.4f}%")
print(f"Final acceptance rate  : {acc_rates[-TAIL:].mean():.3f}")
print(f"Final step size        : {step_sizes[-1]:.4f}")

## Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(
    f"Calogero-Sutherland  N={N_PARTICLES}  L={L}  "
    f"(exact model: single parameter λ → L)",
    fontsize=13,
)
print(len(energies))

ax = axes[0, 0]
ax.plot(steps, energies, lw=0.8, label='E/N')
ax.fill_between(steps, energies - stds, energies + stds, alpha=0.3, label='±σ')
ax.axhline(E_EXACT, color='red', ls='--', lw=1.2, label=f'Exact E₀/N = {E_EXACT:.4f}')
ax.set_xlabel('Epoch'); ax.set_ylabel('E/N'); ax.set_title('Energy'); ax.legend(); ax.set_ylim(E_EXACT, 10)

ax = axes[0, 1]
ax.semilogy(steps, np.abs(energies - E_EXACT), lw=0.8, color='C1')
ax.set_xlabel('Epoch'); ax.set_ylabel('|E/N − E₀/N|'); ax.set_title('Energy error (log)')

ax = axes[0, 2]
ax.semilogy(steps, stds, lw=0.8, color='C2')
ax.set_xlabel('Epoch'); ax.set_ylabel('σ(E)/N'); ax.set_title('Energy std (log)')

if history[-1].params["params"].get("lam") is not None:
    ax = axes[1, 0]
    ax.plot(steps, lambdas, lw=0.8, color='C3', label='λ(t)')
    ax.axhline(L, color='red', ls='--', lw=1.2, label=f'L = {L}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('λ'); ax.set_title('Jastrow exponent λ → L'); ax.legend()

ax = axes[1, 1]
ax.plot(steps, acc_rates, lw=0.8, color='C4')
ax.axhline(0.5, color='red', ls='--', label='target 0.5')
ax.set_xlabel('Epoch'); ax.set_ylabel('Acceptance rate'); ax.set_title('MH acceptance rate'); ax.legend()

ax = axes[1, 2]
ax.plot(steps, step_sizes, lw=0.8, color='C5')
ax.set_xlabel('Epoch'); ax.set_ylabel('Step size'); ax.set_title('MH step size')

plt.tight_layout()
plt.savefig('cs_model_training.png', dpi=150, bbox_inches='tight')
plt.show()